# سبق 11 - ایجنٹ سے ایجنٹ (A2A) پروٹوکول


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A پروٹوکول کیا ہے؟

یہ **ایجنٹ سے ایجنٹ (A2A) پروٹوکول** ایک اوپن اسٹینڈرڈ ہے جو AI ایجنٹس کو باہم بات چیت کرنے، ایک دوسرے کو دریافت کرنے، اور تعاون کرنے کے قابل بناتا ہے — یہاں تک کہ جب وہ مختلف فریم ورک پر بنے ہوں یا مختلف سروسز پر ہوسٹ کیے گئے ہوں۔

اہم تصورات:

- **دریافت** – ایجنٹس ایک *Agent Card* شائع کرتے ہیں جو ان کی صلاحیتوں کو بیان کرتی ہے، جس سے دوسرے ایجنٹس (یا آرکسٹریٹرز) کے لیے کسی کام کے لیے صحیح ماہر تلاش کرنا آسان ہو جاتا ہے۔
- **پیغام رسانی** – ایجنٹس ایک مشترکہ پروٹوکول کے ذریعے منظم شدہ پیغامات کا تبادلہ کرتے ہیں، تاکہ ایک ایجنٹ کی درخواست کو دوسرے ایجنٹ کی داخلی عمل آوری سے قطع نظر سمجھا اور پورا کیا جا سکے۔
- **ٹاسک لائف سائیکل** – A2A ایسے اسٹیٹس کی تعریف کرتا ہے جیسے *submitted*, *working*, *completed*, اور *failed*, جس سے آرکسٹریٹر کو اس بات کی مکمل بصیرت ملتی ہے کہ سونپی گئی ذمہ داری کس طرح آگے بڑھ رہی ہے۔

اس سبق میں ہم A2A انداز کے تعاون کی نقل کرتے ہیں، تین مخصوص سفری ایجنٹس کو ایک ورک فلو میں جوڑ کر جہاں ہر ایجنٹ اپنی مہارت فراہم کرتا ہے اور نتائج اگلے ایجنٹ کو منتقل کرتا ہے۔


## خصوصی سفری ایجنٹس کی تخلیق


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ورک فلو کے ذریعے کثیر-ایجنٹ تعاون

ہم تینوں ایجنٹس کو ایک ترتیب وار ورک فلو میں جوڑتے ہیں جو A2A پیغام رسانی کی عکاسی کرتا ہے:

1. **CurrencyExchangeAgent** صارف کی درخواست وصول کرتا ہے اور کرنسی کی رہنمائی تیار کرتا ہے۔
2. **ActivityPlannerAgent** غنی سیاق و سباق وصول کرتا ہے اور سرگرمیوں کی سفارشات شامل کرتا ہے۔
3. **TravelManagerAgent** دونوں ان پٹس کو یکجا کر کے حتمی سفر کا خلاصہ تیار کرتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## پیداوار میں A2A کی سمجھ

پیداوار کے ماحول میں A2A پروٹوکول طاقتور کراس-سروس منظرنامے کھولتا ہے:

| Capability | Description |
|---|---|
| **فریم ورک کے درمیان باہمی عمل** | ایک فریم ورک سے بنایا گیا ایجنٹ کسی دوسرے A2A کے مطابق فریم ورک میں بنے ایجنٹ کو کام سونپ سکتا ہے، جو حقیقی اداروں کے درمیان باہمی عمل کو ممکن بناتا ہے۔ |
| **سروس کی حدود** | ایجنٹس الگ مائکرو سروسز، کلاؤڈ ریجنز، یا یہاں تک کہ مختلف تنظیموں میں موجود ہو سکتے ہیں اور پھر بھی بغیر کسی رکاوٹ کے باہم تعاون کر سکتے ہیں۔ |
| **متحرک دریافت** | ایک آرکسٹریٹر رن ٹائم پر Agent Card رجسٹری سے استفسار کر سکتا ہے تاکہ دیے گئے ذیلی کام کے لیے بہترین ماہر تلاش کیا جا سکے۔ |
| **اسٹریمنگ & پش نوٹیفیکیشنز** | A2A ریئل ٹائم پیش رفت کی اپ ڈیٹس کے لیے Server-Sent Events (SSE) کی حمایت کرتا ہے اور طویل المدت کاموں کے لیے پش نوٹیفیکیشنز فراہم کرتا ہے۔ |

اوپر جو ورک فلو ہم نے بنایا ہے وہ اس پیٹرن کا ایک سادہ، ان-پروسیس ورژن ہے۔ حقیقی تعیناتی میں ہر ایجنٹ ایک HTTP endpoint ظاہر کرے گا، ایک Agent Card شائع کرے گا، اور A2A JSON-RPC پروٹوکول کے ذریعے بات چیت کرے گا۔


## خلاصہ

اس سبق میں آپ نے سیکھا:

1. **A2A پروٹوکول کیا ہے** — ایجنٹ سے ایجنٹ تک دریافت، پیغام رسانی،  
   اور ٹاسک مینجمنٹ۔
2. **خصوصی ایجنٹس کیسے بنائے جاتے ہیں** — ایک کرنسی ایکسچینج ایجنٹ، ایک ایکٹیویٹی پلانر ایجنٹ،  
   اور ایک ٹریول مینیجر آرکسٹریٹر۔
3. **ایجنٹس کو ورک فلو میں جوڑنے کا طریقہ** — `WorkflowBuilder` کا استعمال کرتے ہوئے ترتیب وار  
   ایجنٹس کے درمیان پیغام رسانی کی ماڈلنگ۔
4. **پروڈکشن میں A2A کیسے کام کرتا ہے** — کراس-فریم ورک، کراس-سروس تعاون کو ممکن بنانا  
   متحرک دریافت اور اسٹریمنگ اپڈیٹس کے ساتھ۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
دستبرداری:
اس دستاویز کا ترجمہ AI ترجمہ سروس "Co-op Translator" (https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ ہم درستگی کے لیے کوشش کرتے ہیں، تاہم براہِ کرم نوٹ کریں کہ خودکار ترجموں میں غلطیاں یا عدمِ درستی ہو سکتی ہے۔ اصل دستاویز کو اس کی مادری زبان میں معتبر ماخذ سمجھنا چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
